# Autoencoder - on the MNIST dataset:

The following is an illustration/exercise in using an Auto-Encoder on image data. The general idea is, that if one can encode (i.e. represent with "few" parameters) images, and then from this encoding decode them again to obtain "almost" the same images, then the few parameters (in a latent space) contains the basic information about the images.

![](AutoEncoder.png "")

An AutoEncoder is trained by requiring that the output images match the input images *best possible*, which is (of course) represented by a loss function. Thus, **an AutoEncoder does not require labels**, and is as such unsupervised learning. The power lies in that one gets a good representation of the data through the latent space (the size of which you can choose) for essentially any NN-based method, here a CNN.

This can then be used (aggressively!) for many things such as:
* Compression (with loss!) of the images (to the size of the latent space)
* De-noising images
* Anomaly detection
* Unsupervised learning (e.g. clustering) on images, sound, graphs, etc.

In the example at hand, we consider the MNIST dataset (28x28 (=784) images of digits), and autoencode these images into a latent space of size 4. We then project this latent space onto just two dimensions using UMAP, and see if we (or a clustering algorithm) can detect clusters and possibly determine how many digits there are... *without ever having known anything about arabic numerals!!!*

***

Last Update: 2026-10-28


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd 
import random 

import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torch import nn
import torch.nn.functional as F
import torch.optim as optim

from umap import UMAP

from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
import itertools

In [ ]:
# Convert data to torch.FloatTensor
transform = transforms.ToTensor()

# Load the training and test datasets
train_dataset = datasets.MNIST('dataset', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST('dataset', train=False, download=True, transform=transform)

In [ ]:
# Split training data into training and validation data:
full_len = len(train_dataset)
train_data, val_data = random_split(train_dataset, [int(full_len-full_len*0.2), int(full_len*0.2)])
batch_size=256

### Question - do we need to split the data?

We somehow always split the data into these three datasets, but given that we are doing unsupervised learning here, try to think about the degree to which we need to divide the data in this way.

In [ ]:
# Create PyTorch dataloaders for data:
train_loader = DataLoader(train_data, batch_size=batch_size)
valid_loader = DataLoader(val_data, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False)

In [ ]:
# Define the autoencoder model (explained in comments below):

class Encoder(nn.Module):
    
    def __init__(self, encoded_space_dim,fc2_input_dim):
        super().__init__()
        
        ### Convolutional section:
        # So we decide to take our 1 input image and first apply 8 kernels/filters of size 3x3
        # with a stride (i.e. step size) of 2 and padding the edge with one layer.
        # This output goes through a ReLU and then through 16 new kernels/filters
        # of the same size, stride and padding, which is then BatchNorm'ed, ReLU'ed, and finally
        # convoluted a third time, again doubling the number of kernels/filters (quite standard!).
        self.encoder_cnn = nn.Sequential(
            nn.Conv2d(1, 8, 3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(8, 16, 3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(True),
            nn.Conv2d(16, 32, 3, stride=2, padding=0),
            nn.ReLU(True)
        )
        
        ### Flatten layer
        # The images/matrices are then flatten'ed (put into one long array), put through a linear layer,
        # then a ReLU, and then finally another linear layer, which boils it down to the latent space
        # dimension.
        self.flatten = nn.Flatten(start_dim=1)
        ### Linear section
        self.encoder_lin = nn.Sequential(
            nn.Linear(3 * 3 * 32, 128),
            nn.ReLU(True),
            nn.Linear(128, encoded_space_dim)
        )
        
    def forward(self, x):
        x = self.encoder_cnn(x)
        x = self.flatten(x)
        x = self.encoder_lin(x)
        return x
    
class Decoder(nn.Module):
    
    def __init__(self, encoded_space_dim,fc2_input_dim):
        super().__init__()
        self.decoder_lin = nn.Sequential(
            nn.Linear(encoded_space_dim, 128),
            nn.ReLU(True),
            nn.Linear(128, 3 * 3 * 32),
            nn.ReLU(True)
        )

        ### The decoder does the exact opposite, reconstructing the images from the latent space values.
        self.unflatten = nn.Unflatten(dim=1, 
        unflattened_size=(32, 3, 3))

        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 3, stride=2, output_padding=0),
            nn.BatchNorm2d(16),
            nn.ReLU(True),
            nn.ConvTranspose2d(16, 8, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(True),
            nn.ConvTranspose2d(8, 1, 3, stride=2, padding=1, output_padding=1)
        )
        
    def forward(self, x):
        x = self.decoder_lin(x)
        x = self.unflatten(x)
        x = self.decoder_conv(x)
        x = torch.sigmoid(x)
        return x

In [ ]:
# Define loss function and training parameters:

# Define the loss function
loss_fn = nn.MSELoss()

# Define an optimizer (both for the encoder and the decoder!)
lr = 0.001

# Set the random seed for reproducible results
torch.manual_seed(42)

# Initialize the two networks
d = 4
encoder = Encoder(encoded_space_dim=d, fc2_input_dim=128)
decoder = Decoder(encoded_space_dim=d, fc2_input_dim=128)
params_to_optimize = [
    {'params': encoder.parameters()},
    {'params': decoder.parameters()}
]

optim = torch.optim.Adam(params_to_optimize, lr=lr, weight_decay=1e-05)

In [ ]:
# Define training and validation functions:

### Training function
def train_epoch(encoder, decoder, dataloader, loss_fn, optimizer):
    # Set train mode for both the encoder and the decoder
    encoder.train()
    decoder.train()
    train_loss = []
    # Iterate the dataloader (we do not need the label values, this is unsupervised learning)
    for image_batch, _ in dataloader: # with "_" we just ignore the labels (the second element of the dataloader tuple)
        # Encode data
        encoded_data = encoder(image_batch)
        # Decode data
        decoded_data = decoder(encoded_data)
        # Evaluate loss
        loss = loss_fn(decoded_data, image_batch)
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # Print batch loss
#         print('\t partial train loss (single batch): %f' % (loss.data))
        train_loss.append(loss.detach().cpu().numpy())

    return np.mean(train_loss)


### Testing function
def test_epoch(encoder, decoder, dataloader, loss_fn):
    # Set evaluation mode for encoder and decoder
    encoder.eval()
    decoder.eval()
    with torch.no_grad(): # No need to track the gradients
        # Define the lists to store the outputs for each batch
        conc_out = []
        conc_label = []
        for image_batch, _ in dataloader:
            # Encode data
            encoded_data = encoder(image_batch)
            # Decode data
            decoded_data = decoder(encoded_data)
            # Append the network output and the original image to the lists
            conc_out.append(decoded_data.cpu())
            conc_label.append(image_batch.cpu())
        # Create a single tensor with all the values in the lists
        conc_out = torch.cat(conc_out)
        conc_label = torch.cat(conc_label) 
        # Evaluate global loss
        val_loss = loss_fn(conc_out, conc_label)
    return val_loss.data


def plot_ae_outputs(encoder,decoder,n=10):
    plt.figure(figsize=(16,4.5))
    targets = test_dataset.targets.numpy()
    t_idx = {i:np.where(targets==i)[0][0] for i in range(n)}
    for i in range(n):
        
        ax = plt.subplot(2,n,i+1)
        img = test_dataset[t_idx[i]][0].unsqueeze(0)
        encoder.eval()
        decoder.eval()
        with torch.no_grad():
            rec_img  = decoder(encoder(img))
        plt.imshow(img.cpu().squeeze().numpy(), cmap='gist_gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)  
        if i == n//2:
            ax.set_title('Original images')
        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(rec_img.cpu().squeeze().numpy(), cmap='gist_gray')  
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)  
        if i == n//2:
            ax.set_title('Reconstructed images')
    plt.show()   

### Question - what is the dimension of the latent space?

Ask yourself, how many dimensions we should reduce the images to, i.e., how many numbers represent what the images look like?

NOTE: You may want to launch the training cell below (which takes a few minutes) and then think about the answer.

In [ ]:
# Train the model
num_epochs = 10
diz_loss = {'train_loss':[],'val_loss':[]}
for epoch in range(num_epochs):
    train_loss = train_epoch(encoder,decoder, train_loader, loss_fn, optim)
    val_loss = test_epoch(encoder,decoder,test_loader,loss_fn)
    print('\n EPOCH {}/{} \t train loss {} \t val loss {}'.format(epoch + 1, num_epochs,train_loss,val_loss))
    diz_loss['train_loss'].append(train_loss)
    diz_loss['val_loss'].append(val_loss)
    plot_ae_outputs(encoder,decoder,n=10)

In [ ]:
# Plot losses over epochs:
plt.figure(figsize=(10,8))
plt.semilogy(diz_loss['train_loss'], label='Train')
plt.semilogy(diz_loss['val_loss'], label='Valid')
plt.xlabel('Epoch')
plt.ylabel('Average Loss')
#plt.grid()
plt.legend()
#plt.title('loss')
plt.show()

### Extract latent space representation

In [ ]:
# Perform forward pass on test set and subtract latent layer (and labels):
latent_test = []
labels_test = []
for step, (images, labels) in enumerate(test_loader): 
    with torch.no_grad():
        latent = encoder(images)
    latent_test.extend(latent.numpy())
    labels_test.extend(labels.numpy())  
    
# labels_str = list(map(str, labels_test))
    
print(f'latent shape: {np.shape(latent_test)}, labels shape: {np.shape(labels_test)}')

## Reduce latent space with UMAP

In [ ]:
# Fit a (possibly parametric!) UMAP embedding to the latent space representation:
# embedder = ParametricUMAP(n_epochs=50, n_neighbors=20, min_dist=0.1)
embedder = UMAP(n_neighbors=50, min_dist=0.2)
embedding = embedder.fit_transform(latent_test)

### Question - can you "train" UMAP on a training set, and then apply the same model to a test set?

Seeing how t-SNE and UMAP iteratively got to their resulting projections, do you think that at the end of the process there is a model, that can be applied to other data?  If not, how one could construct a good approximation?

In [ ]:
# Plot embedding without example images:
fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.tab10
scatter = ax.scatter(embedding[:, 0], embedding[:, 1], c=labels_test, s=10, cmap=cmap)
ax.legend(*scatter.legend_elements())
ax.axis("off");

In [ ]:
# Plot embedding with example images:
fig, ax = plt.subplots(figsize=(15, 10))
cmap = plt.cm.tab10
scatter = ax.scatter(embedding[:, 0], embedding[:, 1], c=labels_test, s=10, cmap=cmap)
ax.legend(*scatter.legend_elements())
image_positions = np.array([[1., 1.]])
for index, position in enumerate(embedding):
    dist = np.sum((position - image_positions) ** 2, axis=1)
    if np.min(dist) > 4:            # If far enough from other images...
        image_positions = np.r_[image_positions, [position]]
        imagebox = mpl.offsetbox.AnnotationBbox(
            mpl.offsetbox.OffsetImage(test_dataset[index][0][0], cmap="binary"),
            position, bboxprops={"edgecolor": cmap(labels_test[index]), "lw": 2})
        fig.gca().add_artist(imagebox)
ax.axis("off");

### Visualizing what the full latent space looks like

In the VAE notebook we plot a regular grid over the 2D latent space and decode every point — because the VAE's Gaussian constraint fills the whole space with meaningful content, the result is a smooth manifold of recognisable digits.

We show two versions here, which tell quite different stories:

**Version 1 — Raw latent space grid.** We pick two latent dimensions (holding the other two fixed at their mean), sweep a regular grid, and decode every point directly. This is the true equivalent of the VAE plot. Most of the grid lands in voids the AE was never trained to fill, so those points decode to blur or noise.

**Version 2 — UMAP nearest-neighbour grid.** We lay the same grid over the 2D UMAP embedding of the real encoded test images, and at each grid point decode the nearest real latent vector. Because we are always snapping to an actual data point, everything looks like a digit — but this is misleading: UMAP is hiding the voids by never letting us visit them.

The contrast between the two plots is the key message: the AE latent space has large empty regions that decode to nothing useful, whereas the VAE's Gaussian prior ensures the whole space is filled.

In [ ]:
import numpy as np
import torch
from PIL import Image
from IPython.display import display

# ── helpers ────────────────────────────────────────────────────────────────

def make_grid_image(rows_of_digits, digit_size=28):
    """Stack a list-of-lists of (28,28) arrays into one big PIL image."""
    n = len(rows_of_digits[0])
    m = len(rows_of_digits)
    figure = np.zeros((m * digit_size, n * digit_size), dtype=np.float32)
    for i, row in enumerate(rows_of_digits):
        for j, digit in enumerate(row):
            figure[i*digit_size:(i+1)*digit_size,
                   j*digit_size:(j+1)*digit_size] = digit
    img = (figure * 255).clip(0, 255).astype(np.uint8)
    return Image.fromarray(img, mode='L')


# ── Version 1: raw latent space grid ───────────────────────────────────────

def plot_ae_raw_grid(decoder, latent_test, dim0=0, dim1=1, n=25, digit_size=28):
    """
    Sweep a grid over two latent dimensions (dim0, dim1), holding the
    remaining dimensions fixed at their mean.  Decode every grid point
    directly — no nearest-neighbour snapping.  Voids in the latent space
    will decode to blur or noise.
    """
    latent_array = np.array(latent_test)           # (N, d)
    mean_latent  = latent_array.mean(axis=0)       # (d,)

    # Set sweep range to ±3 std along the two chosen dimensions
    std0 = latent_array[:, dim0].std()
    std1 = latent_array[:, dim1].std()
    grid0 = np.linspace(-3*std0 + mean_latent[dim0],
                          3*std0 + mean_latent[dim0], n)
    grid1 = np.linspace( 3*std1 + mean_latent[dim1],
                         -3*std1 + mean_latent[dim1], n)  # top-to-bottom

    decoder.eval()
    rows = []
    with torch.no_grad():
        for v1 in grid1:
            row = []
            for v0 in grid0:
                z = torch.tensor(mean_latent, dtype=torch.float32).unsqueeze(0)
                z[0, dim0] = v0
                z[0, dim1] = v1
                digit = decoder(z).squeeze().cpu().numpy()   # (28, 28)
                row.append(digit)
            rows.append(row)

    print(f'Version 1 — raw latent grid (dim {dim0} × dim {dim1}, others fixed at mean)')
    display(make_grid_image(rows, digit_size))


# ── Version 2: UMAP nearest-neighbour grid ─────────────────────────────────

def plot_ae_umap_grid(decoder, latent_test, embedding, n=25, digit_size=28):
    """
    Lay a regular grid over the 2D UMAP embedding of the AE latent space.
    At each grid point decode the nearest real latent vector.
    Because we always snap to a real data point this looks good — but it
    hides the voids that Version 1 reveals.
    """
    latent_array = np.array(latent_test)
    embed_array  = np.array(embedding)

    x_min, x_max = embed_array[:, 0].min(), embed_array[:, 0].max()
    y_min, y_max = embed_array[:, 1].min(), embed_array[:, 1].max()
    pad = 0.05
    xr = x_max - x_min;  yr = y_max - y_min
    x_min -= pad*xr;  x_max += pad*xr
    y_min -= pad*yr;  y_max += pad*yr

    grid_x = np.linspace(x_min, x_max, n)
    grid_y = np.linspace(y_max, y_min, n)   # top-to-bottom

    decoder.eval()
    rows = []
    with torch.no_grad():
        for yi in grid_y:
            row = []
            for xi in grid_x:
                dists  = (embed_array[:, 0] - xi)**2 + (embed_array[:, 1] - yi)**2
                nn_idx = int(np.argmin(dists))
                z = torch.tensor(latent_array[nn_idx], dtype=torch.float32).unsqueeze(0)
                digit = decoder(z).squeeze().cpu().numpy()
                row.append(digit)
            rows.append(row)

    print('Version 2 — UMAP nearest-neighbour grid (always snaps to a real data point)')
    display(make_grid_image(rows, digit_size))


# ── Run both ────────────────────────────────────────────────────────────────
plot_ae_raw_grid(decoder, latent_test, dim0=1, dim1=0, n=25)
plot_ae_umap_grid(decoder, latent_test, embedding, n=25)

In [ ]:
# k-means clustering of embedding:
kmeans = KMeans(n_clusters=10, random_state=42)
kmeans.fit(embedding)
clusters = pd.DataFrame(kmeans.labels_, columns=['Cluster'])

classes = np.array(labels_test)
class_names = [0,1,2,3,4,5,6,7,8,9]

clust0 = clusters.loc[clusters['Cluster'] == 0].index.to_numpy()
clust1 = clusters.loc[clusters['Cluster'] == 1].index.to_numpy()
clust2 = clusters.loc[clusters['Cluster'] == 2].index.to_numpy()
clust3 = clusters.loc[clusters['Cluster'] == 3].index.to_numpy()
clust4 = clusters.loc[clusters['Cluster'] == 4].index.to_numpy()
clust5 = clusters.loc[clusters['Cluster'] == 5].index.to_numpy()
clust6 = clusters.loc[clusters['Cluster'] == 6].index.to_numpy()
clust7 = clusters.loc[clusters['Cluster'] == 7].index.to_numpy()
clust8 = clusters.loc[clusters['Cluster'] == 8].index.to_numpy()
clust9 = clusters.loc[clusters['Cluster'] == 9].index.to_numpy()
cluster_list = [clust0, clust1, clust2, clust3, clust4, clust5, clust6, clust7, clust8, clust9]

real_labels = np.concatenate([classes[clust0], classes[clust1], classes[clust2], classes[clust3], classes[clust4], classes[clust5],
                             classes[clust6], classes[clust7], classes[clust8], classes[clust9]])
pred_labels = []

for cluster in cluster_list:
    unique, pos = np.unique(classes[cluster], return_inverse=True)
    maxpos = np.bincount(pos).argmax()
    cluster_label = unique[maxpos]
    for i in range(len(cluster)):
        pred_labels.append(cluster_label)

pred_labels = np.array(pred_labels)

In [ ]:
# Function to plot confusion matrix:
def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True,
                          savefig = ''):

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(6, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title, fontsize=16)
    cbar = plt.colorbar(fraction=0.046, pad=0.04)
    
    
    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45, fontsize=16)
        plt.yticks(tick_marks, target_names, fontsize=16)
        

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.2f}".format(cm[i, j]),
                     horizontalalignment="center",
                     fontsize=13,
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     fontsize=16,
                     color="white" if cm[i, j] > thresh else "black")


    #plt.tight_layout()
    plt.ylabel('True label', fontsize=16)
    plt.xlabel(f'Predicted label\n\n accuracy={accuracy:0.3f}; misclass={misclass:0.3f}', fontsize=16)
    cbar.ax.set_ylabel('Number of items',  labelpad=20, rotation=270, fontsize=16)   
    
    
    if savefig: plt.savefig(savefig, bbox_inches='tight')
    
    plt.show()

In [ ]:
# Plot confusion matrix:
cm = confusion_matrix(real_labels, pred_labels, labels=class_names)
plot_confusion_matrix(cm, target_names=class_names)

### Question - are you satisfied with the confusion matrix?

Is the performance "good enough"? If not, what do you think is the problem, and how could it be solved in a simple manner? And if this is not enough, discuss what else could be done. 

## Anomaly Detection with the Autoencoder

One of the applications listed in the introduction is **anomaly detection**. The idea is simple:

- Train the AE on a "normal" dataset (here: MNIST digits).
- When you feed it something it has never seen — an anomaly — the reconstruction will be poor, because the encoder has no good latent representation for it.
- The **reconstruction loss** therefore acts as an anomaly score: high loss → likely anomaly.

We can demonstrate this using **Fashion-MNIST** as the anomaly class. The AE was trained only on handwritten digits; Fashion-MNIST images (clothes, shoes, etc.) are a completely different distribution.

The cell below:
1. Loads a batch of Fashion-MNIST images.
2. Computes per-image reconstruction loss for both MNIST digits and Fashion-MNIST images.
3. Plots the two loss distributions side-by-side.

**What to look for:** You should see the Fashion-MNIST losses clustered at higher values than the MNIST losses. The degree of separation tells you how useful the AE is as an anomaly detector.

**Think about it:** What threshold would you set to flag an anomaly? What happens if you reduce the latent space dimension `d`? Does the separation get better or worse, and why?

In [ ]:
from torchvision.datasets import FashionMNIST
import matplotlib.pyplot as plt
import numpy as np
import torch

# Load Fashion-MNIST (anomaly class) - same transforms as MNIST
fashion_dataset = FashionMNIST('dataset', train=False, download=True,
                                transform=transforms.ToTensor())
fashion_loader  = DataLoader(fashion_dataset, batch_size=256, shuffle=False)

def reconstruction_losses(enc, dec, dataloader, loss_fn, max_batches=5):
    """Return per-image MSE reconstruction loss for all images in dataloader."""
    enc.eval()
    dec.eval()
    losses = []
    with torch.no_grad():
        for i, (imgs, _) in enumerate(dataloader):
            if i >= max_batches:
                break
            recon = dec(enc(imgs))
            # Per-image loss: mean over pixels
            per_img = ((recon - imgs) ** 2).mean(dim=(1, 2, 3))
            losses.extend(per_img.cpu().numpy().tolist())
    return np.array(losses)

# Compute losses for MNIST test images and Fashion-MNIST images
mnist_losses   = reconstruction_losses(encoder, decoder, test_loader,   loss_fn)
fashion_losses = reconstruction_losses(encoder, decoder, fashion_loader, loss_fn)

# Plot distributions
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, max(fashion_losses.max(), mnist_losses.max()), 80)
ax.hist(mnist_losses,   bins=bins, alpha=0.6, label='MNIST (normal)',    color='steelblue')
ax.hist(fashion_losses, bins=bins, alpha=0.6, label='Fashion-MNIST (anomaly)', color='tomato')
ax.set_xlabel('Per-image reconstruction loss (MSE)', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Anomaly detection: reconstruction loss distributions', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f'MNIST   losses  — mean: {mnist_losses.mean():.4f},   std: {mnist_losses.std():.4f}')
print(f'Fashion losses  — mean: {fashion_losses.mean():.4f},  std: {fashion_losses.std():.4f}')

# Simple threshold: 2 standard deviations above mean MNIST loss
threshold = mnist_losses.mean() + 2 * mnist_losses.std()
tp_rate = (fashion_losses > threshold).mean() * 100
fp_rate = (mnist_losses   > threshold).mean() * 100
print(f'\nThreshold = {threshold:.4f}  (MNIST mean + 2 std)')
print(f'  Fashion-MNIST correctly flagged as anomaly: {tp_rate:.1f}%')
print(f'  MNIST incorrectly flagged as anomaly:       {fp_rate:.1f}%')

### Final questions - could you learn the number of digits?

Imagine that you didn't know the digits (or something else like zebra calls!), and wanted to estimate this. Can you think of how to do that, and would you be sure to get the right result or a reasonable estimate of it?

## Learning Points:
From this exercise, you should:
1. Have understood how Auto-Encoders work, and why they can be used for unsupervised learning.
2. Know that AEs can be used for all types of data that NNs operate on.
3. Know that AEs consist of an encoder and a decoder, and compares input to output as loss function.
4. See the difference between supervised and unsupervised learning in this case (and in general).
5. Understand how AEs can be used for anomaly detection.
6. Be able to apply this or similar algorithms to other datasets, images or not.